# Intro: why compile?
By default, torch executes operations eagerly.  It computes `sin` over all of `x`, then `cos` over all of `y`, and so on

In [2]:
import torch

In [2]:
def f(x):
    y = x.sin()
    z = y.cos()
    return z

f(torch.randn(3, 3))

tensor([[0.6236, 0.5622, 0.9402],
        [0.7330, 0.6591, 0.9227],
        [0.9934, 0.9422, 0.5491]])

What happens if we instead compile this function?

In [3]:
torch._logging.set_logs(graph_code=True)

comp_f = torch.compile(f)
comp_f(torch.randn(3, 3))

V0928 12:45:44.869000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code] TRACED GRAPH
V0928 12:45:44.869000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code]  ===== __compiled_fn_1_25707943_46f0_48f7_8ca7_df7d3fd02cfc =====
V0928 12:45:44.869000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V0928 12:45:44.869000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V0928 12:45:44.869000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code]         l_x_ = L_x_
V0928 12:45:44.869000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code] 
V0928 12:45:44.869000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code]         # File: /tmp/ipykernel_15673/1201477647.py:2 in f, code: y = x.sin()
V0928 12:45:44.869000 15673 torc

tensor([[0.5404, 0.5536, 0.8323],
        [0.9908, 0.9555, 0.5415],
        [0.7133, 0.9497, 0.9975]])

First we figure out what the function is doing, storing the results in a format called an "FX graph".  Then we can compile the grpah to executable code.  I'm running on CPU, so this gives the following C++ code (you can see this for yourself by running with `TORCH_COMPILE_DEBUG=1` and looking in the `output_code.py` file) 

```c++
#include <torch/csrc/inductor/cpp_prefix.h>
extern "C"  void  kernel(const float* in_ptr0,
                       float* out_ptr0)
{
    std::atomic<int> inductor_cpu_integer_div_error{0};
    inductor_cpu_integer_div_error_flag = &inductor_cpu_integer_div_error;
    {
        for(int64_t x0=static_cast<int64_t>(0L); x0<static_cast<int64_t>(9L); x0+=static_cast<int64_t>(8L))
        {
            {
                if(C10_LIKELY(x0 >= static_cast<int64_t>(0) && x0 < static_cast<int64_t>(8L)))
                {
                    auto tmp0 = at::vec::Vectorized<float>::loadu(in_ptr0 + static_cast<int64_t>(x0), static_cast<int64_t>(8));
                    auto tmp1 = tmp0.sin();
                    auto tmp2 = tmp1.cos();
                    tmp2.store(out_ptr0 + static_cast<int64_t>(x0));
                }
                if(C10_UNLIKELY(x0 >= static_cast<int64_t>(8L) && x0 < static_cast<int64_t>(9L)))
                {
                    auto tmp0 = at::vec::Vectorized<float>::loadu(in_ptr0 + static_cast<int64_t>(x0), static_cast<int64_t>(1L));
                    auto tmp1 = tmp0.sin();
                    auto tmp2 = tmp1.cos();
                    tmp2.store(out_ptr0 + static_cast<int64_t>(x0), static_cast<int64_t>(1L));
                }
            }
        }
    }
    inductor_cpu_integer_div_error_flag = nullptr;
    inductor_cpu_throw_if_integer_div_error(inductor_cpu_integer_div_error);
}
```

Note that it's vectorized, and that there are fewer memory accesses.  This matters even more on GPU!

# But how does it work?
If we only cared about compiling code that looks like this, torch.compile could be a lot simpler.  We could just track whenever a torch operation occurs and add it to the FX graph.  But what if we have code like this?

In [4]:
num_calls = 0

@torch.compile(backend='eager')
def f(x):
    global num_calls
    num_calls += 1
    y = x.sin()
    z = y.cos()
    return z

torch._logging.set_logs(graph_code=True, side_effects=True, bytecode=True)
f(torch.randn(3, 3))

V0928 12:45:54.570000 15673 torch/_dynamo/convert_frame.py:1770] [1/0] [__bytecode] ORIGINAL BYTECODE f /tmp/ipykernel_15673/167820838.py line 3 
V0928 12:45:54.570000 15673 torch/_dynamo/convert_frame.py:1770] [1/0] [__bytecode]   3           RESUME                   0
V0928 12:45:54.570000 15673 torch/_dynamo/convert_frame.py:1770] [1/0] [__bytecode] 
V0928 12:45:54.570000 15673 torch/_dynamo/convert_frame.py:1770] [1/0] [__bytecode]   6           LOAD_GLOBAL              0 (num_calls)
V0928 12:45:54.570000 15673 torch/_dynamo/convert_frame.py:1770] [1/0] [__bytecode]               LOAD_SMALL_INT           1
V0928 12:45:54.570000 15673 torch/_dynamo/convert_frame.py:1770] [1/0] [__bytecode]               BINARY_OP               13 (+=)
V0928 12:45:54.570000 15673 torch/_dynamo/convert_frame.py:1770] [1/0] [__bytecode]               STORE_GLOBAL             0 (num_calls)
V0928 12:45:54.570000 15673 torch/_dynamo/convert_frame.py:1770] [1/0] [__bytecode] 
V0928 12:45:54.570000 15673 to

tensor([[0.8986, 0.9855, 0.5691],
        [0.6461, 0.6228, 0.6045],
        [0.9762, 0.9601, 0.8184]])

Now our code has a side effect.  FX graphs are optimized for torch operations - they don't include non-torch side effects.  Note that the graph is the same, but we've capture the side effect separately.  We'd have no way of knowing this just by looking at which torch operators are called - we need to symbolically execute the function and see what it does.  This is the job of Dynamo.  It produces an FX graph which is sent to the other major component of torch.compile - Inductor, which generates executable code.

### Graph breaks
This isn't such a hard side effect, but python is a big language and dynamo can't understand all of it.  What happens then?

In [5]:
import pickle

@torch.compile(backend='eager')
def f(x):
    y = x.sin()
    pickle.dumps('something')
    z = y.cos()
    return z

torch._logging.set_logs(graph_code=True)
f(torch.randn(3, 3))

/home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/_dynamo/variables/functions.py:2882: UserWarning: Dynamo does not know how to trace the builtin `_pickle.dumps.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround.
If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use `torch.compiler.allow_in_graph`.
  torch._dynamo.utils.warn_once(explanation + "\n" + "\n".join(hints))
V0928 12:45:56.795000 15673 torch/_dynamo/output_graph.py:2967] [2/0_1] [__graph_code] TRACED GRAPH
V0928 12:45:56.795000 15673 torch/_dynamo/output_graph.py:2967] [2/0_1] [__graph_code]  ===== _

tensor([[0.8810, 0.8247, 0.6665],
        [0.7180, 0.5686, 0.7650],
        [0.9767, 0.9345, 0.9022]])

Notice that we now have two graphs instead of one.  Dynamo started to trace our function, then got to something it didn't understand (pickle).  This is a graph break - we stop capturing the current graph, then fall back to the normal python interpreter to execute the unknown portion, then resume tracing.  This is a very flexible approach, but it comes with a performance cost - we can no longer optimize between the two graphs.  We can use `fullgraph=True` to detect when this happens - this will error out instead of silently graph breaking.

In [6]:
@torch.compile(backend='eager', fullgraph=True)
def f(x):
    y = x.sin()
    pickle.dumps('something')
    z = y.cos()
    return z

f(torch.randn(3, 3))

Unsupported: Attempted to call function marked as skipped
  Explanation: Dynamo does not know how to trace the builtin `_pickle.dumps.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
  Hint: If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround.
  Hint: If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use `torch.compiler.allow_in_graph`.

  Developer debug context: module: _pickle, qualname: dumps, skip reason: cannot determine source file for _pickle (likely a C extension or builtin)

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0007.html

from user code:
   File "/tmp/i

There are many reasons that Dynamo might graph break.

- Python features that Dynamo doesn't support
- Native code
- Fundamental (I/O, data-dependent control flow, etc.)

Notably, the last category includes constructs that are not supported by FX graphs.  If your code includes these, you will need to either accept the graph breaks, or devote some effort to avoiding them.

### Recompiles
Recompiles are another problem you might run into.

In [21]:
@torch.compile(backend='eager')
def f(x, flag):
    if flag:
        return x.sin()
    return x.cos()

torch._logging.set_logs(graph_code=True)
f(torch.randn(3, 3), True)

V0928 12:48:47.545000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code] TRACED GRAPH
V0928 12:48:47.545000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code]  ===== __compiled_fn_26_608afd57_6039_4589_9870_a6398bab46b1 =====
V0928 12:48:47.545000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V0928 12:48:47.545000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V0928 12:48:47.545000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code]         l_x_ = L_x_
V0928 12:48:47.545000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code] 
V0928 12:48:47.545000 15673 torch/_dynamo/output_graph.py:2967] [0/0] [__graph_code]         # File: /tmp/ipykernel_15673/2688769548.py:4 in f, code: return x.sin()
V0928 12:48:47.545000 15673 

tensor([[ 0.6824, -0.9597,  0.2275],
        [ 0.9022,  0.2929, -0.4346],
        [ 0.2507, -0.6875, -0.9995]])

Notice that there is no control flow in the graph - we've baked in the value of `flag`.  What if its value changes?

In [22]:
torch._logging.set_logs(graph_code=True, guards=True)

f(torch.randn(3, 3), False)

V0928 12:48:48.810000 15673 torch/_dynamo/output_graph.py:2967] [0/1] [__graph_code] TRACED GRAPH
V0928 12:48:48.810000 15673 torch/_dynamo/output_graph.py:2967] [0/1] [__graph_code]  ===== __compiled_fn_28_01e87621_c791_499a_99a7_375cb50de2b7 =====
V0928 12:48:48.810000 15673 torch/_dynamo/output_graph.py:2967] [0/1] [__graph_code]  /home/rob/code/scratch/pytorch/presentation/env/lib/python3.14/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
V0928 12:48:48.810000 15673 torch/_dynamo/output_graph.py:2967] [0/1] [__graph_code]     def forward(self, L_x_: "f32[3, 3][3, 1]cpu"):
V0928 12:48:48.810000 15673 torch/_dynamo/output_graph.py:2967] [0/1] [__graph_code]         l_x_ = L_x_
V0928 12:48:48.810000 15673 torch/_dynamo/output_graph.py:2967] [0/1] [__graph_code] 
V0928 12:48:48.810000 15673 torch/_dynamo/output_graph.py:2967] [0/1] [__graph_code]         # File: /tmp/ipykernel_15673/2688769548.py:5 in f, code: return x.cos()
V0928 12:48:48.810000 15673 

tensor([[ 0.3149,  0.7996,  0.9992],
        [-0.0253,  0.7232,  0.9233],
        [ 0.2836,  0.6206, -0.1086]])

We need to recompile `f` and make a new FX graph.  Dynamo handles this using guards: a series of conditions which are checked before the compiled function is run.  If any guard fails, the function is recompiled.

Another common source of recompiles is changing tensor shapes

In [8]:
@torch.compile(backend='eager')
def f(x):
    return x.cos()

torch._logging.set_logs(recompiles=True)
f(torch.randn(3, 3))
f(torch.randn(3, 4))

tensor([[-0.1704,  0.3837,  0.9633,  0.9810],
        [ 0.0447,  0.7998,  0.9738,  0.8207],
        [ 0.4475,  0.8967, -0.0905, -0.2161]])

Dynamo has a number of tools to work around this.  The simplest is to treat all shapes as dynamic

In [12]:
@torch.compile(backend='eager', dynamic=True)
def f(x):
    return x.cos()

torch._logging.set_logs(recompiles=True)
f(torch.randn(3, 4))
f(torch.randn(3, 5))

tensor([[0.8978, 0.7713, 0.9954, 0.9256, 0.8690],
        [0.5146, 0.3694, 0.2947, 0.9812, 0.9792],
        [0.6427, 0.9999, 0.7186, 0.9152, 0.9546]])

# Troubleshooting

With the basics covered, let's talk about troubleshooting more specific problems

### Working around graph breaks

Graph breaks are a common source of problems.  Let's see how to work around some common sources.



In [16]:
@torch.compile(backend='eager', fullgraph=True)
def f(x):
    y = x.sin()
    print(y)
    return y.cos()

f(torch.ones(3, 3))

Unsupported: Failed to trace builtin operator
  Explanation: Dynamo does not know how to trace builtin operator `print` with argument types ['Tensor'] (has_kwargs False)
  Hint: Avoid calling builtin `print` with argument types ['Tensor']. Consider using an equivalent alternative function/method to `print`.
  Hint: If you are attempting to call a logging function (e.g. `print`), you can try adding it to `torch._dynamo.config.reorderable_logging_functions`.
  Hint: Please report an issue to PyTorch.

  Developer debug context: builtin print [<class 'torch._dynamo.variables.tensor.TensorVariable'>] False

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0059.html

from user code:
   File "/tmp/ipykernel_15542/2137740994.py", line 4, in f
    print(y)
    ~~~~~^^^

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set T

Some options

- reorderable_logging_functions
- ignorable_logging_functions
- print HOP

### Recompiles

- logging function changing f-string

### Dynamic shapes

- discuss different tools for marking shapes dynamic

### Escape hatches

- custom ops
- nonstrict trace
- disable

### Higher order ops

More dynamo resources:

- [Overview](https://docs.pytorch.org/docs/2.14/user_guide/torch_compiler/torch.compiler_dynamo_overview.html)
- [Deep dive](https://docs.pytorch.org/docs/2.14/user_guide/torch_compiler/torch.compiler_dynamo_deepdive.html)
- [Troubleshooting](https://docs.pytorch.org/docs/2.14/user_guide/torch_compiler/torch.compiler_troubleshooting.html)